# Assignment II – Bike Sharing Demand Prediction
**GitHub Repository:** Add Link

**Author:** Alesia Dako
**Date:** May 2026

## Task 1: Exploratory Data Analysis (EDA)

In this section, we explore the dataset to understand the structure of the data, the distribution of the target variable, and the relationships between features and bike rental counts. This analysis guides our feature engineering and modeling decisions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

from skopt import BayesSearchCV
from skopt.space import Real, Integer

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load dataset
df = pd.read_csv("data/hour.csv")

# Basic info
print(f"Dataset Shape: {df.shape}")
df.head()

### 1.1 Dataset Overview

We begin by inspecting the shape, data types, and first rows of the dataset to understand what we are working with. The dataset contains 17,379 hourly records with no missing values, which simplifies preprocessing.

In [ ]:
# Target distribution
df['cnt'].describe()
print(f"Skewness: {df['cnt'].skew():.3f}")
plt.figure(figsize=(8,4))
sns.histplot(df['cnt'], kde=True)
plt.title("Distribution of cnt")
plt.show()

### 1.2 Dropping Irrelevant Columns

We drop `instant` (a row index with no predictive value), `dteday` (redundant since temporal information is already captured by `hr`, `mnth`, `yr`), and `casual` and `registered` (they are components of the target variable `cnt` and would cause data leakage if kept as features).

In [ ]:
# Drop irrelevant columns
df.drop(columns=['instant', 'dteday', 'casual', 'registered'], inplace=True)

### 1.3 Target Variable Distribution

The target variable `cnt` represents the total number of bike rentals per hour. Its distribution is right-skewed, meaning most hours have relatively low rental counts, with a long tail of high-demand hours. This skewness is expected in demand data and suggests that tree-based models may handle it better than linear ones, which assume normally distributed residuals.

In [ ]:
# Temporal patterns
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flatten(), ['hr', 'weekday', 'mnth', 'season']):
    df.groupby(col)['cnt'].mean().plot(ax=ax, marker='o')
    ax.set_title(f'Avg cnt by {col}')
plt.tight_layout()
plt.show()

### Analytical Reasoning on Distribution:
- **Skewness:** The target variable cnt has a skewness of approximately 1.27 , confirming a strong right-skewed distribution. This suggests that while linear models might face challenges with non-normal residuals, tree-based models like Random Forest and Gradient Boosting are well-suited for this data.

- **Outliers:** The boxplot identifies several high-demand outliers. However, these are valid data points representing peak usage hours in Washington D.C. and will be retained to maintain the integrity of the business prediction problem. 

In [ ]:
# Skeweness
cnt_skew = df['cnt'].skew()
print(f"Target Variable Skewness: {cnt_skew:.3f}")

plt.figure(figsize=(10, 2))
sns.boxplot(x=df['cnt'])
plt.title("Boxplot of Bike Rental Count (cnt) for Outlier Identification")
plt.show()